# Settings

In [232]:
# import requests
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.remote.webelement import WebElement
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

from bs4 import BeautifulSoup
import pandas as pd
import json


In [42]:
def get_all_classes(element: WebElement) -> list:
    """
    Extracts all unique class names from a Selenium WebElement.

    Args:
        element (WebElement): The Selenium WebElement to extract classes from.

    Returns:
        list: A sorted list of unique class names found in the element.
    """
    html = element.get_attribute("innerHTML")  # Extract inner HTML
    soup = BeautifulSoup(html, 'html.parser')

    all_classes = set()
    for el in soup.find_all(class_=True):  # Find all elements that have a class
        all_classes.update(el["class"])  # Add all class names to the set

    return sorted(all_classes)  # Return sorted list for consistency

In [230]:
def clean_dict(d):
    return {k: v for k, v in d.items() if v and str(v).strip()}

# Run

In [241]:
import platform
print(platform.architecture())

('64bit', 'WindowsPE')


In [242]:
# Set up Selenium WebDriver
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

# Open Website
url = 'https://www.zonaprop.com.ar/departamentos-venta-villa-del-parque-villa-devoto-1-ambiente.html'

driver.get(url)
time.sleep(5)  # Wait for the page to load

### Data

In [251]:
scroll_box = driver.find_element(By.CLASS_NAME, "postingsList-module__postings-container")

property_listings = scroll_box.find_elements(By.CLASS_NAME, "postingsList-module__card-container")  # Update with the actual class name

len(property_listings)

30

In [ ]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 10)  # Wait up to 10 seconds
cards = wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "postingCard-module__posting-card-row")))
html_list = [card.get_attribute("innerHTML") for card in cards]
soup = BeautifulSoup(html_list[0], 'html.parser')

In [252]:
html = property_listings[0].get_attribute("innerHTML")
html

'<div class="postingCardLayout-module__posting-card-layout" data-qa="posting PROPERTY" data-id="55621355" data-posting-type="PROPERTY" data-to-posting="/propiedades/clasificado/veclapin-venta-monoamb-balcon-aterrazado-villa-del-parque-55621355.html"><div class="postingCardLayout-module__posting-card-container"><div class="postingGallery-module__gallery-container postingGallery-module__container-width postingGallery-module__tablet-width" data-qa="POSTING_CARD_GALLERY"><div class="lazyload-wrapper " style="width:100%;height:100%;position:absolute"><div class="gallery-container "><div class="multimediaGallery flickity-enabled is-draggable" tabindex="0"><div class="flickity-viewport" style="height: 259.869px; touch-action: pan-y;"><div class="flickity-slider" style="left: 0px; transform: translateX(0%);"><img fetchpriority="high" loading="eager" alt="Departamento · 35m² · 1 Ambiente · Venta Monoamb, Balcón Aterrazado, Villa del Parque" width="100%" height="100%" src="https://imgar.zonaprop

### Funcion Final

In [287]:
soup.find('span', class_=['postingMainFeatures-module__posting-main-features-span', 'postingMainFeatures-module__posting-main-features-listing']).text

'48 un.'

In [301]:
def extract_info(html):
    soup = BeautifulSoup(html, 'html.parser')
    
    data_element = soup.find(attrs={"data-id": True})
    anchor_element = soup.find("a")

    squared_meters = soup.find('span', class_='postingMainFeatures-module__posting-main-features-span postingMainFeatures-module__posting-main-features-listing').text
    
    location = soup.find('h2', class_='postingLocations-module__location-text').get_text(strip=True)
    # adress = soup.find('div', class_='postingLocations-module__location-address postingLocations-module__location-address-in-listing').get_text(strip=True)
    
    # adress = soup.find('span', class_=['postingLocations-module__location-address','postingLocations-module__location-address-in-listing']).text
    rooms = soup.find('span', class_=['postingMainFeatures-module__posting-main-features-span', 'postingMainFeatures-module__posting-main-features-listing']).text
    # data_property_type = soup.find(attrs={"data-posting-type": True})
        
    description = soup.find(attrs={"data-qa": "POSTING_CARD_DESCRIPTION"}).get_text(strip=True) if soup.find(attrs={"data-qa": "POSTING_CARD_DESCRIPTION"}) else None

    property_info = {
        "id": data_element.get("data-id") if data_element else None,
        "property_type": data_element.get("data-posting-type") if data_element else None,
        
        "price": (price_data := soup.find(attrs={"data-qa": "POSTING_CARD_PRICE"})) and price_data.get_text(strip=True),
        "squared_meters": squared_meters,
        "rooms": (rooms := soup.find("span", class_=['postingMainFeatures-module__posting-main-features-span', 'postingMainFeatures-module__posting-main-features-listing'])) and rooms.get_text(strip=True),
        
        
               
        "title": (title := soup.find("h2")) and title.get_text(strip=True),
        "address": (address := soup.find("div", class_=['postingLocations-module__location-address','postingLocations-module__location-address-in-listing'])) and address.get_text(strip=True),       
        "description":  description,
        "image_url": (img := soup.find("img")) and img.get("src"),
        "property_url": 'https://www.zonaprop.com.ar'+ data_element.get("data-to-posting") if data_element else None,
        "listing_url": "https://www.zonaprop.com.ar" + anchor_element.get("href") if anchor_element and anchor_element.get("href") else None,

    }
    
    return property_info



In [302]:
data = []
for i in range(30):
   data.append(extract_info(html_list[i]))

In [305]:
df = pd.DataFrame(data)
df.head(4)

,id,property_type,price,squared_meters,rooms,title,address,description,image_url,property_url,listing_url
0,55810290,PROPERTY,USD 78.000,46 m² tot.,46 m² tot.,"Villa del Parque, Capital Federal",Cuenca y Elpidio Gonzáles,_ Departamento monoambiente en venta ubicado e...,https://imgar.zonapropcdn.com/avisos/1/00/55/8...,https://www.zonaprop.com.ar/propiedades/clasif...,https://www.zonaprop.com.ar/propiedades/clasif...
1,54709539,PROPERTY,USD 99.500,35 m² tot.,35 m² tot.,"Villa del Parque, Capital Federal",Av. Nazca al 2800,Monoambientecocina americana balconcalefaccion...,https://imgar.zonapropcdn.com/avisos/1/00/54/7...,https://www.zonaprop.com.ar/propiedades/clasif...,https://www.zonaprop.com.ar/propiedades/clasif...
2,55608553,PROPERTY,USD 69.900,34 m² tot.,34 m² tot.,"Villa del Parque, Capital Federal",Tres Arroyos 3000,Corredor Responsable: Ariel Champanier cucicba...,https://imgar.zonapropcdn.com/avisos/1/00/55/6...,https://www.zonaprop.com.ar/propiedades/clasif...,https://www.zonaprop.com.ar/propiedades/clasif...
3,52569639,DEVELOPMENT,USD 93.300,48 un.,48 un.,"Villa del Parque, Capital Federal",Nogoyá 3865,Villa del Parque fusiona la serenidad caracter...,https://imgar.zonapropcdn.com/avisos/1/00/52/5...,https://www.zonaprop.com.ar/propiedades/empren...,https://www.zonaprop.com.ar/propiedades/empren...


# Normalizador

In [306]:
pip install usig-normalizador-amba

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for usig-normalizador-amba: filename=usig_normalizador_amba-1.3.0-py3-none-any.whl size=19536 sha256=1231334c5a69de96bd607bda3f13eac078eb5bce8c5cc251cf899d93a76b3820
  Stored in directory: c:\users\juani\appdata\local\pip\cache\wheels\20\da\ee\ccdefd0cbc425f334e5d7e455ac33b804be6a746fa5469c09b
Successfully built usig-normalizador-amba
Note: you may need to restart the kernel to use updated packages.


In [323]:
res[0].partido.nombre

'CABA'

In [325]:
res[0].calle.nombre

'TRES ARROYOS'

In [326]:
res[0].altura

3000

In [327]:
test = 'Tres Arroyos 3000'
test = 'Cuenca y Elpidio Gonzáles'
res = nd.normalizar(test)
for r in res:
            print('Partido: ', res.partido.nombre)
            print('Localidad: ', res.localidad)
            print('Nombre de la calle: ', res.calle.nombre)
            print('Altura: ', res.altura)
            print('___________________')

ErrorCruceInexistente: Cruce inexistente: CUENCA y ELPIDIO GONZALES

In [ ]:
for i in df.address:
    print('A normalizar en un partido (caba): ',i)
    try:
        res = nd.normalizar(i)
        for r in res:
            print('Partido: ', res[cont].partido.nombre)
            print('Localidad: ', res[cont].localidad)
            print('Nombre de la calle: ', res[cont].calle.nombre)
            print('Altura: ', res[cont].altura)
            print('___________________')
    except:
        pass

A normalizar en un partido (caba):  Cuenca y Elpidio Gonzáles
A normalizar en un partido (caba):  Av. Nazca  al 2800
A normalizar en un partido (caba):  Tres Arroyos 3000
A normalizar en un partido (caba):  Nogoyá 3865
A normalizar en un partido (caba):  Nogoya 3235
A normalizar en un partido (caba):  Cervantes al 2100
A normalizar en un partido (caba):  Tres Arroyos 3000
A normalizar en un partido (caba):  Camarones 2733
A normalizar en un partido (caba):  Jose Pedro Varela 4400
A normalizar en un partido (caba):  Gualeguaychu 3000
A normalizar en un partido (caba):  AV. Nazca 2800
A normalizar en un partido (caba):  Av. San Martin 4820
A normalizar en un partido (caba):  Gabriela Mistral al 3300
A normalizar en un partido (caba):  Nazca 2600
A normalizar en un partido (caba):  Avda. Francisco Beiró 4200
A normalizar en un partido (caba):  Avenida Beiró al 5000
A normalizar en un partido (caba):  Simbron 3214
A normalizar en un partido (caba):  Villa Devoto
A normalizar en un partido 

In [307]:
from usig_normalizador_amba import NormalizadorAMBA
nd = NormalizadorAMBA(include_list=['caba']) # lista de codigos de partido
str = u'San Martín 153'
cont=0
print('A normalizar en un partido (caba): ',str)
try:
    res = nd.normalizar(str)
    for r in res:
        print('Partido: ', res[cont].partido.nombre)
        print('Localidad: ', res[cont].localidad)
        print('Nombre de la calle: ', res[cont].calle.nombre)
        print('Altura: ', res[cont].altura)
        print('___________________')
        cont += 1
except Exception as e:
    print('error')
    print('___________________')

A normalizar en un partido (caba):  San Martín 153
Partido:  CABA
Localidad:  CABA
Nombre de la calle:  SAN MARTIN
Altura:  153
___________________


#### Function 1 extract_property_info

In [ ]:
def extract_property_info(html):
    soup = BeautifulSoup(html, 'html.parser')
    
    data_element = soup.find(attrs={"data-id": True})
    anchor_element = soup.find("a")
    
    property_info = {
        "id": data_element.get("data-id") if data_element else None,
        "title": (title := soup.find("h2")) and title.get_text(strip=True),
        "image_url": (img := soup.find("img")) and img.get("src"),
        "price": (price := soup.find("span", class_="price-tag")) and price.get_text(strip=True),
        "address": (address := soup.find("span", class_="address")) and address.get_text(strip=True),
        "location": (location := soup.find("span", class_="location")) and location.get_text(strip=True),
        "size": (size := soup.find("span", class_="size")) and size.get_text(strip=True),
        "rooms": (rooms := soup.find("span", class_="rooms")) and rooms.get_text(strip=True),
        "bathrooms": (bathrooms := soup.find("span", class_="bathrooms")) and bathrooms.get_text(strip=True),
        "features": [feature.get_text(strip=True) for feature in soup.find_all("span", class_="feature")] or [],
        "listing_url": "https://www.zonaprop.com.ar" + anchor_element.get("href") if anchor_element and anchor_element.get("href") else None,
        "publisher": (publisher := soup.find("span", class_="publisher")) and publisher.get_text(strip=True),
        "highlight": (highlight := soup.find("span", class_="highlight")) and highlight.get_text(strip=True),
    }
    
    return property_info


#### Function 2 extract_zonaprop_details

In [ ]:
def extract_zonaprop_details(html):
    soup = BeautifulSoup(html, 'html.parser')
    
    # Extract property ID (usually in a script tag or meta tag)
    property_id_tag = soup.find('script', {'type': 'application/ld+json'})
    property_id = None
    if property_id_tag:
        try:
            data = json.loads(property_id_tag.string)
            property_id = data.get('identifier')
        except json.JSONDecodeError:
            pass
    
    # Extract title
    title_tag = soup.find('h1')
    title = title_tag.text.strip() if title_tag else None
    
    # Extract price
    price_tag = soup.find('div', class_='price')
    price = price_tag.text.strip() if price_tag else None
    
    # Extract location
    location_tag = soup.find('span', class_='location')
    location = location_tag.text.strip() if location_tag else None
    
    # Extract images
    images = []
    image_tags = soup.find_all('img')
    for img in image_tags:
        img_url = img.get('src')
        if img_url and 'http' in img_url:
            images.append(img_url)
    
    return {
        'property_id': property_id,
        'title': title,
        'price': price,
        'location': location,
        'images': images
    }

In [45]:
details = extract_zonaprop_details(html)
details

{'property_id': None,
 'title': None,
 'price': None,
 'location': None,
 'images': ['https://imgar.zonapropcdn.com/avisos/1/00/55/81/02/90/360x266/1965249634.jpg?isFirstImage=true',
  'https://imgar.zonapropcdn.com/avisos/1/00/55/81/02/90/360x266/1965249621.jpg',
  'https://img10.naventcdn.com/listado/RPLISv8.185.2-RC1/images/tagsSpriteNew.e033f42a.png',
  'https://img10.naventcdn.com/listado/RPLISv8.185.2-RC1/images/tagsSpriteNew.e033f42a.png',
  'https://imgar.zonapropcdn.com/empresas/1/00/17/16/10/84/130x70/logo_yacoub_1722526533223.jpg']}

#### Function 3

In [37]:
from selenium.webdriver.remote.webelement import WebElement

def extract_property_details(elements: list[WebElement]) -> dict:
    """
    Extracts property details like price, expenses, total area, rooms, and bathrooms
    from the provided WebElement list.
    
    Args:
        elements (list[WebElement]): List of WebElements containing price and property details.
        
    Returns:
        dict: Extracted details as a structured dictionary.
    """
    property_details = {}
    
    for element in elements:
        # Extract price
        price_elem = element.find_element("xpath", ".//*[@data-qa='POSTING_CARD_PRICE']")
        if price_elem:
            property_details["price"] = price_elem.text.strip()
        
        # Extract expenses
        expenses_elem = element.find_elements("xpath", ".//*[@data-qa='expensas']")
        if expenses_elem:
            property_details["expenses"] = expenses_elem[0].text.strip()
        
        # Extract main features
        features_elem = element.find_elements("xpath", ".//*[@data-qa='POSTING_CARD_FEATURES']//span")
        if features_elem:
            features = [feature.text.strip() for feature in features_elem]
            if len(features) >= 3:
                property_details["total_area"] = features[0]
                property_details["rooms"] = features[1]
                property_details["bathrooms"] = features[2]
    
    return property_details



In [46]:

card = p.find_elements(By.CLASS_NAME, "postingCard-module__posting-card-row")  # Update with the actual class name
card

StaleElementReferenceException: Message: stale element reference: stale element not found in the current frame
  (Session info: chrome=134.0.6998.119); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#stale-element-reference-exception
Stacktrace:
	GetHandleVerifier [0x00007FF7D0EC4C25+3179557]
	(No symbol) [0x00007FF7D0B288A0]
	(No symbol) [0x00007FF7D09B91CA]
	(No symbol) [0x00007FF7D09C0B98]
	(No symbol) [0x00007FF7D09C3BCC]
	(No symbol) [0x00007FF7D09C3C9F]
	(No symbol) [0x00007FF7D0A0F320]
	(No symbol) [0x00007FF7D0A0FC9C]
	(No symbol) [0x00007FF7D0A028FC]
	(No symbol) [0x00007FF7D0A37C6F]
	(No symbol) [0x00007FF7D0A022D6]
	(No symbol) [0x00007FF7D0A37E40]
	(No symbol) [0x00007FF7D0A602F3]
	(No symbol) [0x00007FF7D0A37A03]
	(No symbol) [0x00007FF7D0A006D0]
	(No symbol) [0x00007FF7D0A01983]
	GetHandleVerifier [0x00007FF7D0F267CD+3579853]
	GetHandleVerifier [0x00007FF7D0F3D1D2+3672530]
	GetHandleVerifier [0x00007FF

In [16]:
price_card = card[0]
price_card_classes = get_all_classes(price_card)

price_card_data = {}
for k in price_card_classes:
    temp = property_listings[0].find_elements(By.CLASS_NAME, k)[0]
    price_card_data[k] = temp.text.strip()

def clean_dict(d):
    return {k: v for k, v in d.items() if v and str(v).strip()}


clean_dict(price_card_data)

{'postingCard-module__posting-prices-and-publisher': 'USD 78.000\n$ 45.000 Expensas',
 'postingPrices-module__expenses': '$ 45.000 Expensas',
 'postingPrices-module__expenses-property-listing': '$ 45.000 Expensas',
 'postingPrices-module__posting-card-price-block': 'USD 78.000\n$ 45.000 Expensas',
 'postingPrices-module__price': 'USD 78.000',
 'postingPrices-module__price-container': 'USD 78.000'}

In [20]:
price_card = card[1]
price_card_classes = get_all_classes(price_card)

price_card_data = {}
for k in price_card_classes:
    temp = property_listings[0].find_elements(By.CLASS_NAME, k)[0]
    price_card_data[k] = temp.text.strip()




clean_dict(price_card_data)

{'postingMainFeatures-module__posting-main-features-block': '46 m² tot.\n1 amb.\n1 baño',
 'postingMainFeatures-module__posting-main-features-block-one-line': '46 m² tot.\n1 amb.\n1 baño',
 'postingMainFeatures-module__posting-main-features-listing': '46 m² tot.',
 'postingMainFeatures-module__posting-main-features-span': '46 m² tot.'}

In [ ]:
price_card = get_all_classes(card[0])


In [34]:
property_listings[0]

<selenium.webdriver.remote.webelement.WebElement (session="3bbb7aca08aeac6b1bc869addcb671e0", element="f.437BF87ED6EA807585C6B094E6152A9E.d.6D557C1825DC6C2C9ECBF9A336B1FBFC.e.140")>

In [33]:
class_ls = get_all_classes(property_listings[0])

StaleElementReferenceException: Message: stale element reference: stale element not found in the current frame
  (Session info: chrome=134.0.6998.119); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#stale-element-reference-exception
Stacktrace:
	GetHandleVerifier [0x00007FF7D0EC4C25+3179557]
	(No symbol) [0x00007FF7D0B288A0]
	(No symbol) [0x00007FF7D09B91CA]
	(No symbol) [0x00007FF7D09C0B98]
	(No symbol) [0x00007FF7D09C3F11]
	(No symbol) [0x00007FF7D0A61652]
	(No symbol) [0x00007FF7D0A37C2A]
	(No symbol) [0x00007FF7D0A602F3]
	(No symbol) [0x00007FF7D0A37A03]
	(No symbol) [0x00007FF7D0A006D0]
	(No symbol) [0x00007FF7D0A01983]
	GetHandleVerifier [0x00007FF7D0F267CD+3579853]
	GetHandleVerifier [0x00007FF7D0F3D1D2+3672530]
	GetHandleVerifier [0x00007FF7D0F32153+3627347]
	GetHandleVerifier [0x00007FF7D0C9092A+868650]
	(No symbol) [0x00007FF7D0B32FFF]
	(No symbol) [0x00007FF7D0B2F4A4]
	(No symbol) [0x00007FF7D0B2F646]
	(

In [22]:
results = {}
for k in class_ls:
    temp = property_listings[0].find_elements(By.CLASS_NAME, k)[0]
    results[k] = temp.text.strip()

res = {}
for k,v in results.items():
    if v=='':
        pass
    else:
        res[k] = v

In [ ]:
html = test[1].get_attribute("innerHTML")  # Extract inner HTML
html

In [ ]:
{
'postingCard-module__posting-card-row': 'price_1',
'postingCard-module__posting-container': "full",
'postingCard-module__posting-description': 'description',
'postingLocations-module__location-address':"adress_1",
'postingLocations-module__location-address-in-listing':'adress_2',
'postingLocations-module__location-block':'adress_3',
'postingPrices-module__price': 'price_2',
 'postingPrices-module__price-container': 'price_3',
}

In [ ]:
'postingCard-module__posting-card-row'

In [ ]:
for klas in all_classes:
    try:
        temp = p.find_elements(By.CLASS_NAME, klas)[0]
        print(f"{klas} :")
        print(f"{temp.text}")
              
    except:
    
        print(f"couldn't find {klas}")
              

In [ ]:
temp = "postingCardLayout-module__posting-card-layout"
test = p.find_elements(By.CLASS_NAME, temp)

html = test[0].get_attribute("innerHTML")  # Extract inner HTML
html

In [ ]:
p.find_elements(By.CLASS_NAME, 'postingMainFeatures-module__posting-main-features-block')

In [ ]:
if type(p) == selenium.webdriver.remote.webelement.WebElement :
    print('yes')

In [ ]:
type(p)

In [ ]:
# Parse with BeautifulSoup
soup = BeautifulSoup(html, 'html.parser')

# Find all elements with a class attribute
all_classes = set()
for element in soup.find_all(class_=True):  # Find all elements that have a class
    all_classes.update(element["class"])  # Add all class names to the set

# Print all unique class names
print(all_classes)

In [ ]:
def extract_property_info(html):
    soup = BeautifulSoup(html, 'html.parser')
    
    property_info = {
        "id": soup.find(attrs={"data-id": True})["data-id"] if soup.find(attrs={"data-id": True}) else None,
        "title": soup.find("h2").get_text(strip=True) if soup.find("h2") else None,
        "image_url": soup.find("img")["src"] if soup.find("img") else None,
        "price": soup.find("span", class_="price-tag").get_text(strip=True) if soup.find("span", class_="price-tag") else None,
        "address": soup.find("span", class_="address").get_text(strip=True) if soup.find("span", class_="address") else None,
        "location": soup.find("span", class_="location").get_text(strip=True) if soup.find("span", class_="location") else None,
        "size": soup.find("span", class_="size").get_text(strip=True) if soup.find("span", class_="size") else None,
        "rooms": soup.find("span", class_="rooms").get_text(strip=True) if soup.find("span", class_="rooms") else None,
        "bathrooms": soup.find("span", class_="bathrooms").get_text(strip=True) if soup.find("span", class_="bathrooms") else None,
        "features": [feature.get_text(strip=True) for feature in soup.find_all("span", class_="feature")],
        "listing_url": "https://www.zonaprop.com.ar" + soup.find("a")["href"] if soup.find("a") else None,
        "publisher": soup.find("span", class_="publisher").get_text(strip=True) if soup.find("span", class_="publisher") else None,
        "highlight": soup.find("span", class_="highlight").get_text(strip=True) if soup.find("span", class_="highlight") else None,
    }
    
    return property_info

# Example usage with your HTML snippet:
# html_snippet = """YOUR_HTML_HERE"""  # Replace with actual HTML
property_data = extract_property_info(html)
print(property_data)

In [ ]:
property_data

In [ ]:
html = p.get_attribute("innerHTML")  # Extract inner HTML
soup = BeautifulSoup(html, "html.parser")
listings = soup.find_all("div", class_="postingLocations-module__location-address")  # Adjust tag if necessary
price =  soup.find_elements(By.CLASS_NAME, 'postingPrices-module__price')[0]
for listing in listings:
    print(listing.text.strip())  # Print extracted addresses
    print(price.text.strip())

In [ ]:
html

In [ ]:
html = scroll_box.get_attribute("innerHTML")  # Extract inner HTML
soup = BeautifulSoup(html, "html.parser")
listings = soup.find_all("div", class_="postingLocations-module__location-address")  # Adjust tag if necessary

for listing in listings:
    print(listing.text.strip())  # Print extracted addresses

In [ ]:
property_listings[0].find_elements(By.CLASS_NAME, "postingCard-module__posting-top")[0]  

In [ ]:
postingLocations-module__location-address postingLocations-module__location-address-in-listing

In [ ]:
domicilio = 'postingLocations-module__location-address postingLocations-module__location-address-in-listing'
test.find_elements(By.CLASS_NAME,domicilio)

In [ ]:
domicilio_class = "postingLocations-module__location-address.postingLocations-module__location-address-in-listing"  
addresses = test.find_elements(By.CSS_SELECTOR, domicilio_class)
for address in addresses:
    print(address.text)


In [ ]:
price = test.find_elements(By.CLASS_NAME, 'postingPrices-module__price')[0].text

In [ ]:
test = property_listings[0].find_elements(By.CLASS_NAME, "postingCard-module__posting-top")[0]  # Update with the actual class name
test.find_elements(By.CLASS_NAME, 'postingPrices-module__price')  

In [ ]:


# Get all property listings inside the container
property_listings = scroll_box.find_elements(By.CLASS_NAME, "postingsList-module__card-container")  # Update with the actual class name

# Extract data from each listing
for listing in property_listings:
    title = listing.find_element(By.CLASS_NAME, "some-title-class").text  # Update class
    price = listing.find_element(By.CLASS_NAME, "some-price-class").text  # Update class
    location = listing.find_element(By.CLASS_NAME, "some-location-class").text  # Update class

    print(f"Title: {title}, Price: {price}, Location: {location}")



# Nuevo

In [ ]:
html = scroll_box.get_attribute("innerHTML")  # Extract inner HTML
soup = BeautifulSoup(html, "html.parser")
listings = soup.find_all("div", class_="postingLocations-module__location-address")  # Adjust tag if necessary

for listing in listings:
    print(listing.text.strip())  # Print extracted addresses


# Martes

In [151]:
html = property_listings[0].get_attribute("innerHTML")

In [ ]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 10)  # Wait up to 10 seconds
cards = wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "postingCard-module__posting-card-row")))
html_list = [card.get_attribute("innerHTML") for card in cards]
soup = BeautifulSoup(html_list[0], 'html.parser')

In [71]:
cards = wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "postingsList-module__card-container")))
html_list = [card.get_attribute("innerHTML") for card in cards]


In [73]:
html_list[0]

'<div class="postingCardLayout-module__posting-card-layout" data-qa="posting PROPERTY" data-id="55810290" data-posting-type="PROPERTY" data-to-posting="/propiedades/clasificado/veclapin-departamento-monoambiente-en-venta-1-bano-46-55810290.html"><div class="postingCardLayout-module__posting-card-container"><div class="postingGallery-module__gallery-container postingGallery-module__container-width postingGallery-module__tablet-width" data-qa="POSTING_CARD_GALLERY"><div class="lazyload-wrapper " style="width: 100%; height: 100%; position: absolute;"><div class="gallery-container "><div class="multimediaGallery flickity-enabled is-draggable" tabindex="0"><div class="flickity-viewport" style="height: 259.869px; touch-action: pan-y;"><div class="flickity-slider" style="left: 0px; transform: translateX(0%);"><img fetchpriority="high" loading="eager" alt="Departamento · 36m² · 1 Ambiente · Departamento Monoambiente en Venta - 1 Baño - 46 m² - Villa del Parque" width="100%" height="100%" src="

In [121]:
soup.select_one('span.postingLocations-module__location-address.postingLocations-module__location-address-in-listing')


In [155]:
soup.find('span', class_=['postingLocations-module__location-address', 'postingLocations-module__location-address-in-listing'])


In [156]:
print(soup.find_all('span'))

[<span class="pills-module__trigger-pill-item-span">Parrilla</span>, <span class="postingMultimediaTags-module__tag-item postingMultimediaTags-module__only-first-element"><span class="postingMultimediaTags-module__tag-info">27</span><img class="postingMultimediaTags-module__tag-icon postingMultimediaTags-module__tag-with-icon" src="https://img10.naventcdn.com/listado/RPLISv8.185.2-RC1/images/tagsSpriteNew.e033f42a.png" style="object-position: 0px 1px;"/></span>, <span class="postingMultimediaTags-module__tag-info">27</span>, <span class="postingMultimediaTags-module__tag-item postingMultimediaTags-module__only-first-element"><img class="postingMultimediaTags-module__tag-icon postingMultimediaTags-module__tag-with-icon" src="https://img10.naventcdn.com/listado/RPLISv8.185.2-RC1/images/tagsSpriteNew.e033f42a.png" style="object-position: 0px -16px;"/></span>, <span class="postingMainFeatures-module__posting-main-features-span postingMainFeatures-module__posting-main-features-listing">46 m

In [158]:
rooms = soup.find('span', class_=['postingMainFeatures-module__posting-main-features-span', 'postingMainFeatures-module__posting-main-features-listing'])
rooms.text

'46 m² tot.'

In [159]:
soup.find('span', class_='postingLocations-module__location-address postingLocations-module__location-address-in-listing')

In [160]:
rooms = soup.find('span', class_='postingMainFeatures-module__posting-main-features-span postingMainFeatures-module__posting-main-features-listing')

In [161]:
soup.select_one('span.postingMainFeatures-module__posting-main-features-span.postingMainFeatures-module__posting-main-features-listing').text

'46 m² tot.'

In [162]:
soup.select_one('postingLocations-module__location-address postingLocations-module__location-address-in-listing')

In [163]:
soup.text

'+19 fotosParrilla27USD 78.000$ 45.000 ExpensasCuenca y Elpidio GonzálesVilla del Parque, Capital Federal46 m² tot.1 amb.1 baño_ Departamento monoambiente en venta ubicado en calle Cuenca y Elpidio Gonzáles (Villa del Parque, Ciudad Autónoma de Buenos Aires). Desarrollado en piso alto por escalera, con disposición contra frente y orientación al norte. _ El mismo posee cocina independiente equipada con muebles sobre y bajo mesada, un espacio principal con espacios de guardado empotrados y un baño completo. Luminoso, cuenta con equipo de aire acondicionado, balcón, lavadero y patio con parrilla. _ La propiedad tiene una superficie total de aproximadamente 46mts2, de los cuales 36mts2 son cubiertos. _ Las expensas al día 10/07/2024 son de aproximadamente $45. 000. Yacoub | Construimos ConfianzaInnovación | Calidad | Sustentabilidad | Diseño70 Mil Metros Cuadrados | 19 Torres La información descripta en el presente aviso es meramente orientativa y no forma parte de ningún tipo de documenta

In [164]:
rooms = soup.select_one('span.postingMainFeatures-module__posting-main-features-span.postingMainFeatures-module__posting-main-features-listing')
rooms.text

'46 m² tot.'

In [167]:
html_list[1]

'<div class="postingCardLayout-module__posting-card-layout" data-qa="posting PROPERTY" data-id="54709539" data-posting-type="PROPERTY" data-to-posting="/propiedades/clasificado/veclapin-venta-monoambiente-c-balcon-villa-del-parque-54709539.html"><div class="postingCardLayout-module__posting-card-container"><div class="postingGallery-module__gallery-container postingGallery-module__container-width postingGallery-module__tablet-width" data-qa="POSTING_CARD_GALLERY"><div class="lazyload-wrapper " style="width: 100%; height: 100%; position: absolute;"><div class="gallery-container "><div class="multimediaGallery flickity-enabled is-draggable" tabindex="0"><div class="flickity-viewport" style="height: 259.869px; touch-action: pan-y;"><div class="flickity-slider" style="left: 0px; transform: translateX(0%);"><img fetchpriority="high" loading="eager" alt="Departamento · 33m² · 1 Ambiente · Venta - Monoambiente C/balcón - Villa del Parque" width="100%" height="100%" src="https://imgar.zonaprop

In [183]:
soup.find('div', class_='postingLocations-module__location-address postingLocations-module__location-address-in-listing').get_text(strip=True)

'Cuenca y Elpidio Gonzáles'

In [186]:
soup.find('div', class_='postingPrices-module__price').get_text(strip=True)

'USD 78.000'

In [211]:
soup.find('h2', class_='postingLocations-module__location-text').get_text(strip=True)

'Villa del Parque, Capital Federal'

In [ ]:
soup.find(attrs={"data-qa": "POSTING_CARD_DESCRIPTION"}).get_text(strip=True)

'_ Departamento monoambiente en venta ubicado en calle Cuenca y Elpidio Gonzáles (Villa del Parque, Ciudad Autónoma de Buenos Aires). Desarrollado en piso alto por escalera, con disposición contra frente y orientación al norte. _ El mismo posee cocina independiente equipada con muebles sobre y bajo mesada, un espacio principal con espacios de guardado empotrados y un baño completo. Luminoso, cuenta con equipo de aire acondicionado, balcón, lavadero y patio con parrilla. _ La propiedad tiene una superficie total de aproximadamente 46mts2, de los cuales 36mts2 son cubiertos. _ Las expensas al día 10/07/2024 son de aproximadamente $45. 000. Yacoub | Construimos ConfianzaInnovación | Calidad | Sustentabilidad | Diseño70 Mil Metros Cuadrados | 19 Torres La información descripta en el presente aviso es meramente orientativa y no forma parte de ningún tipo de documentación contractual. Los datos enunciados fueron proporcionados por los propietarios y pueden arrojar inexactitudes, las superfic

In [200]:
soup.find(attrs={"data-qa": "POSTING_CARD_PRICE"}).get_text(strip=True)

'USD 78.000'

In [199]:
soup.find(attrs={"data-qa": "POSTING_CARD_PRICE"}).get_text(strip=True) if soup.find(attrs={"data-qa": "POSTING_CARD_PRICE"}) else None


'USD 78.000'

In [216]:
precio = soup.find(attrs={"data-qa": "POSTING_CARD_PRICE"}).get_text(strip=True) if soup.find(attrs={"data-qa": "POSTING_CARD_PRICE"}) else None
precio

'USD 78.000'

AttributeError: 'NoneType' object has no attribute 'get_text'

In [168]:
# 1. Price
price = soup.find('div', class_='postingPrices-module__price').get_text(strip=True)

# 2. Location
location = soup.find('h2', class_='postingLocations-module__location-text').get_text(strip=True)

# 3. Description
description = soup.find('h3', class_='postingCard-module__posting-description').get_text(strip=True)

# 4. Main image URL (First image in the gallery)
image_url = soup.find('img', {'fetchpriority': 'high'})['src']

# 5. Property URL (Link to the property details)
property_url = soup.find('a', href=True)['href']

# Output the extracted details
print(f"Price: {price}")
print(f"Location: {location}")
print(f"Description: {description}")
print(f"Image URL: {image_url}")
print(f"Property URL: {property_url}")

Price: USD 78.000
Location: Villa del Parque, Capital Federal
Description: _ Departamento monoambiente en venta ubicado en calle Cuenca y Elpidio Gonzáles (Villa del Parque, Ciudad Autónoma de Buenos Aires). Desarrollado en piso alto por escalera, con disposición contra frente y orientación al norte. _ El mismo posee cocina independiente equipada con muebles sobre y bajo mesada, un espacio principal con espacios de guardado empotrados y un baño completo. Luminoso, cuenta con equipo de aire acondicionado, balcón, lavadero y patio con parrilla. _ La propiedad tiene una superficie total de aproximadamente 46mts2, de los cuales 36mts2 son cubiertos. _ Las expensas al día 10/07/2024 son de aproximadamente $45. 000. Yacoub | Construimos ConfianzaInnovación | Calidad | Sustentabilidad | Diseño70 Mil Metros Cuadrados | 19 Torres La información descripta en el presente aviso es meramente orientativa y no forma parte de ningún tipo de documentación contractual. Los datos enunciados fueron propor

In [106]:
    soup = BeautifulSoup(html, 'html.parser')
    
    data_element = soup.find(attrs={"postingMainFeatures-module__posting-main-features-span postingMainFeatures-module__posting-main-features-listing": True})
    data_element

In [109]:
soup.find('span', class_='postingMainFeatures-module__posting-main-features-span postingMainFeatures-module__posting-main-features-listing').text


'33 m² tot.'

In [107]:
soup.find('span', attrs={'class': 'postingMainFeatures-module__posting-main-features-span postingMainFeatures-module__posting-main-features-listing'})


<span class="postingMainFeatures-module__posting-main-features-span postingMainFeatures-module__posting-main-features-listing">33 m² tot.</span>

In [105]:
extract_info(html_list[3])

{'id': '52569639',
 'property_type': 'DEVELOPMENT',
 'link_to_posting': 'https://www.zonaprop.com.ar/propiedades/emprendimiento/ememvein-1-a-3-ambientes-amenities-villa-del-parque-52569639.html',
 'description': None,
 'title': 'Villa del Parque, Capital Federal',
 'image_url': 'https://imgar.zonapropcdn.com/avisos/1/00/52/56/96/39/720x532/1962081494.jpg?isFirstImage=true',
 'price': None,
 'address': None,
 'location': None,
 'size': None,
 'rooms': None,
 'bathrooms': None,
 'features': [],
 'listing_url': 'https://www.zonaprop.com.ar/propiedades/emprendimiento/ememvein-1-a-3-ambientes-amenities-villa-del-parque-52569639.html',
 'publisher': None,
 'highlight': None}

# Next Button

In [218]:
# Function to click the "Next" button and check if the page changes
def click_next_and_check():
    try:
        # Find the 'Next' button
        next_button = driver.find_element(By.CSS_SELECTOR, 'a[data-qa="PAGING_NEXT"]')

        # Store the current URL before clicking
        current_url = driver.current_url

        # Click the 'Next' button
        next_button.click()

        # Wait for the page to load (you may need to adjust the wait time depending on the page load speed)
        time.sleep(3)

        # Check if the URL changed
        if driver.current_url != current_url:
            print("Moved to the next page!")
        else:
            print("Already on the last page or no more pages to navigate.")
            return False  # Indicate that there's no next page
        
        return True  # Indicate that the page changed

    except Exception as e:
        print("Error:", e)
        return False



In [224]:
# Close the cookie consent banner (adjust the selector accordingly)
try:
    cookie_banner = driver.find_element(By.CSS_SELECTOR, '.CookiesPolicyBanner-module__innerBox')
    close_button = cookie_banner.find_element(By.XPATH, '//button[contains(text(),"Accept")]')  # or a similar button to close
    close_button.click()
except:
    pass #print("Cookie banner not found or already closed.")

# Wait for the 'Next' button to be clickable
try:
    next_button = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, 'a[data-qa="PAGING_NEXT"]'))
    )
    next_button.click()
except:
    print("Next button is not clickable.")


Next button is not clickable.


In [226]:
# Switch to the iframe containing the challenge
iframe = driver.find_element(By.ID, 'cf-chl-widget-mjvwj')
driver.switch_to.frame(iframe)

NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":"[id="cf-chl-widget-mjvwj"]"}
  (Session info: chrome=134.0.6998.119); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x00007FF7D0EC4C25+3179557]
	(No symbol) [0x00007FF7D0B288A0]
	(No symbol) [0x00007FF7D09B91CA]
	(No symbol) [0x00007FF7D0A0FA67]
	(No symbol) [0x00007FF7D0A0FC9C]
	(No symbol) [0x00007FF7D0A63627]
	(No symbol) [0x00007FF7D0A37C6F]
	(No symbol) [0x00007FF7D0A602F3]
	(No symbol) [0x00007FF7D0A37A03]
	(No symbol) [0x00007FF7D0A006D0]
	(No symbol) [0x00007FF7D0A01983]
	GetHandleVerifier [0x00007FF7D0F267CD+3579853]
	GetHandleVerifier [0x00007FF7D0F3D1D2+3672530]
	GetHandleVerifier [0x00007FF7D0F32153+3627347]
	GetHandleVerifier [0x00007FF7D0C9092A+868650]
	(No symbol) [0x00007FF7D0B32FFF]
	(No symbol) [0x00007FF7D0B2F4A4]
	(No symbol) 

In [225]:
# Find the checkbox and click it (assumes it's an actual checkbox inside the iframe)
checkbox = driver.find_element(By.CSS_SELECTOR, 'input[type="checkbox"]')
checkbox.click()

NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":"input[type="checkbox"]"}
  (Session info: chrome=134.0.6998.119); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x00007FF7D0EC4C25+3179557]
	(No symbol) [0x00007FF7D0B288A0]
	(No symbol) [0x00007FF7D09B91CA]
	(No symbol) [0x00007FF7D0A0FA67]
	(No symbol) [0x00007FF7D0A0FC9C]
	(No symbol) [0x00007FF7D0A63627]
	(No symbol) [0x00007FF7D0A37C6F]
	(No symbol) [0x00007FF7D0A602F3]
	(No symbol) [0x00007FF7D0A37A03]
	(No symbol) [0x00007FF7D0A006D0]
	(No symbol) [0x00007FF7D0A01983]
	GetHandleVerifier [0x00007FF7D0F267CD+3579853]
	GetHandleVerifier [0x00007FF7D0F3D1D2+3672530]
	GetHandleVerifier [0x00007FF7D0F32153+3627347]
	GetHandleVerifier [0x00007FF7D0C9092A+868650]
	(No symbol) [0x00007FF7D0B32FFF]
	(No symbol) [0x00007FF7D0B2F4A4]
	(No symbol) [0x0

In [ ]:
# Example of clicking and checking multiple times (looping through pages)
while click_next_and_check():
    print("Continuing to next page...")
    time.sleep(2)  # Optional sleep to prevent excessive requests

# Random Mouse Movements

In [245]:
import time
from selenium.webdriver.common.action_chains import ActionChains
import random

# Find an element to move the cursor over
element = driver.find_element(By.TAG_NAME, 'body')

# Simulate human-like mouse movement
action = ActionChains(driver)
action.move_to_element_with_offset(element, 5, 5).perform()

# Random delays
time.sleep(2 + (random.random() * 3))

In [246]:
import time
import random
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By

def random_mouse_movement(driver, min_delay=2, max_delay=5, move_range=10):
    """
    Simulates human-like random mouse movements.
    
    Args:
        driver (webdriver): The Selenium WebDriver instance.
        min_delay (int, optional): Minimum delay between movements. Default is 2 seconds.
        max_delay (int, optional): Maximum delay between movements. Default is 5 seconds.
        move_range (int, optional): Maximum pixel range for random movements. Default is 10 pixels.
    """
    try:
        element = driver.find_element(By.TAG_NAME, 'body')
        action = ActionChains(driver)

        # Random movement offsets
        x_offset = random.randint(-move_range, move_range)
        y_offset = random.randint(-move_range, move_range)

        action.move_to_element_with_offset(element, x_offset, y_offset).perform()

        # Random delay
        time.sleep(random.uniform(min_delay, max_delay))

    except Exception as e:
        print(f"Error in mouse movement: {e}")



In [247]:
random_mouse_movement(driver)


In [248]:
import time
import random
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

def random_mouse_movement(driver, min_delay=2, max_delay=5, move_range=10, scroll_range=200):
    """
    Simulates human-like random mouse movements and scrolling.

    Args:
        driver (webdriver): The Selenium WebDriver instance.
        min_delay (int, optional): Minimum delay between actions. Default is 2 seconds.
        max_delay (int, optional): Maximum delay between actions. Default is 5 seconds.
        move_range (int, optional): Maximum pixel range for random movements. Default is 10 pixels.
        scroll_range (int, optional): Maximum number of pixels to scroll up/down. Default is 200 pixels.
    """
    try:
        element = driver.find_element(By.TAG_NAME, 'body')
        action = ActionChains(driver)

        # Random movement offsets
        x_offset = random.randint(-move_range, move_range)
        y_offset = random.randint(-move_range, move_range)
        action.move_to_element_with_offset(element, x_offset, y_offset).perform()

        # Random delay
        time.sleep(random.uniform(min_delay, max_delay))

        # Random scrolling (up or down)
        scroll_direction = random.choice([Keys.ARROW_DOWN, Keys.ARROW_UP])
        for _ in range(random.randint(1, 3)):  # Random number of scrolls
            element.send_keys(scroll_direction)
            time.sleep(random.uniform(0.5, 1.5))  # Small delay between scrolls

    except Exception as e:
        print(f"Error in mouse movement and scrolling: {e}")


In [250]:
random_mouse_movement(driver)
